In [ ]:
# Import necessary libraries
import gmsh
import math
import numpy as np
import torch.nn as nn
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
class Generator:
    def __init__(self, domain_length, domain_height, n_elements_x, n_elements_y, generator_type, filename, CNN_params=None):
        self.domain_length = domain_length
        self.domain_height = domain_height
        self.n_elements_x = n_elements_x
        self.n_elements_y = n_elements_y
        self.generator_type = generator_type
        self.filename = filename
        self.hx_objects = []
        self.mesh_min = min(domain_length / n_elements_x, domain_height / n_elements_y)
        self.mesh_max = max(domain_length / n_elements_x, domain_height / n_elements_y)
        self.mesh_params = {'mesh_algorithm': 8, 'mesh_recombine': 1, 'mesh_element_order': 2}
        if generator_type == 'cnn':
            self.cnn_generator = self.initialize_cnn_generator(CNN_params)

    def initialize_cnn_generator(self, CNN_params):
        # Define a simple CNN architecture for mesh generation
        class CNNGenerator(nn.Module):
            def __init__(self):
                super(CNNGenerator, self).__init__()
                self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
                self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
                self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
                self.fc1 = nn.Linear(64 * (self.n_elements_x // 8) * (self.n_elements_y // 8), 128)
                self.fc2 = nn.Linear(128, self.n_elements_x * self.n_elements_y)

                # if weights are provided, load them
                if CNN_params and 'weights' in CNN_params:
                    self.load_state_dict(CNN_params['weights'])

            def forward(self, x):
                x = F.relu(self.conv1(x))
                x = F.max_pool2d(x, 2)
                x = F.relu(self.conv2(x))
                x = F.max_pool2d(x, 2)
                x = F.relu(self.conv3(x))
                x = F.max_pool2d(x, 2)
                x = x.view(-1, 64 * (self.n_elements_x // 8) * (self.n_elements_y // 8))
                x = F.relu(self.fc1(x))
                x = torch.sigmoid(self.fc2(x))
                return x.view(-1, self.n_elements_x, self.n_elements_y)

        return CNNGenerator()
    
    def generate_geometry(self):
        # use cnn generator to determine solid and fluid regions of the geometry
        input_tensor = torch.zeros((1, 1, self.n_elements_x, self.n_elements_y))  # Placeholder input
        output = self.cnn_generator(input_tensor)
        solid_mask = output > 0.5  # Threshold to determine solid vs fluid regions
        return solid_mask
    
    def create_custom_obstruction(self, point_coords):
        # Use the provided coordinates to create points in Gmsh
        n_points = len(point_coords)
        points = []
        for coords in point_coords:
            point = gmsh.model.occ.addPoint(coords[0], coords[1], 0)
            points.append(point)
        
        # Connect the points with lines to form the geometry of the obstruction
        lines = []
        for i in range(n_points):
            p1 = points[i]
            p2 = points[(i + 1) % n_points]  # Wrap around to connect the last point to the first
            line = gmsh.model.occ.addLine(p1, p2)
            lines.append(line)

        # Close the loop to create a surface
        cl = gmsh.model.occ.addCurveLoop(lines)
        surface = gmsh.model.occ.addPlaneSurface([cl])
        return surface

    def generate_mesh(self):
        # Initialize Gmsh and create a new model
        gmsh.initialize()
        gmsh.model.add("heat_exchanger")

        # Use OpenCASCADE kernel
        gmsh.model.occ

        # Define the domain geometry
        rect = gmsh.model.occ.addRectangle(0, -self.height/2, 0, self.length, self.height)
        gmsh.model.occ.synchronize()

        # Add obstructions to the geometry
        obstructions = []
        for obj in self.hx_objects:
            if obj['type'] == 'polygon':
                obstruction = self.create_polygon_obstruction(obj['center'][0], obj['center'][1], obj['radius'], obj['n_pts'])
            elif obj['type'] == 'rectangle':
                obstruction = self.create_rectangle_obstruction(obj['center'], obj['width'], obj['height'])
            elif obj['type'] == 'disk':
                obstruction = self.create_disk_obstruction(obj['center'], obj['radius'])
            elif obj['type'] == 'custom':
                obstruction = self.create_custom_obstruction(obj['coords'])
            obstructions.append(obstruction)
        gmsh.model.occ.synchronize()

        # Boolean cut
        cut = gmsh.model.occ.cut(
            [(2, rect)],
            [(2, s) for s in obstructions],
            removeObject=True,
            removeTool=True
        )
        gmsh.model.occ.synchronize()

        # Extract surfaces
        outDimTags, _ = cut
        fluid_surfaces = [tag for dim, tag in outDimTags if dim == 2]

        if not fluid_surfaces:
            raise RuntimeError("Boolean cut failed")

        gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

        print("Cut result:", outDimTags)
        print("2D entities:", gmsh.model.getEntities(2))
        
        # Cylinder wall curves
        gmsh.model.occ.synchronize()
        boundaries = gmsh.model.getBoundary(
            [(2, s) for s in fluid_surfaces],
            oriented=False
        )

        obstruction_wall_curves = []
        inlet = []
        outlet = []
        top = []
        bottom = []

        for dim, tag in boundaries:
            com = gmsh.model.occ.getCenterOfMass(dim, tag)
            x, y, _ = com

            if abs(x) < 1e-6:
                inlet.append(tag)
            elif abs(x - self.length) < 1e-6:
                outlet.append(tag)
            elif abs(y - self.height/2) < 1e-6:
                top.append(tag)
            elif abs(y + self.height/2) < 1e-6:
                bottom.append(tag)
            else:
                obstruction_wall_curves.append(tag)

        gmsh.model.addPhysicalGroup(1, obstruction_wall_curves, name="Wall")
        gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
        gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
        gmsh.model.addPhysicalGroup(1, top, name="Top")
        gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")
        
        # Set mesh parameters
        gmsh.option.setNumber("Mesh.CharacteristicLengthMin", self.mesh_min)
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", self.mesh_max)
        gmsh.option.setNumber("Mesh.Algorithm", self.mesh_params['mesh_algorithm']) # 8 
        gmsh.option.setNumber("Mesh.RecombineAll", self.mesh_params['mesh_recombine']) # 1
        gmsh.option.setNumber("Mesh.ElementOrder", self.mesh_params['mesh_element_order']) # 2

        # Generate the mesh
        gmsh.model.mesh.generate(2)
        gmsh.write(self.filename)
        gmsh.finalize()




